In [1]:
import pandas as pd
import numpy as np


In [9]:
import os

print(os.getcwd())


/Users/khushimishra/Desktop/India-Trend-Radar


In [11]:
import pandas as pd

news = pd.read_csv("data/processed/news_with_ner.csv")

google = pd.read_csv("data/cleaned/google_trends_clean.csv")

youtube = pd.read_csv("data/cleaned/youtube_clean.csv")


In [12]:
news.head()


,source_name,author,title,description,url,keyword,language,published_date,collected_at,year,month,day,weekday,entities,entity_count
0,GNews,Unknown,india dispatch delhis cockroach janta party pr...,india dispatch delhis cockroach janta party pr...,https://news.google.com/rss/articles/CBMiqAFBV...,"india, dispatch, delhis",en,2026-07-24 04:06:45,2026-07-24 04:06:45,2026,7,24,Friday,"india (GPE), juristorg india (ORG), juristorg ...",3
1,GNews,Unknown,returning to play for india kept me motivated ...,returning to play for india kept me motivated ...,https://news.google.com/rss/articles/CBMipgFBV...,"returning, play, india",en,2026-07-24 04:06:45,2026-07-24 04:06:45,2026,7,24,Friday,"india (GPE), india (GPE)",2
2,GNews,Unknown,us unveils fresh tariffs of up to 125 on 60 ec...,us unveils fresh tariffs of up to 125 on 60 ec...,https://news.google.com/rss/articles/CBMimgJBV...,"india, unveils, fresh",en,2026-07-24 04:06:45,2026-07-24 04:06:45,2026,7,24,Friday,"up to 125 (CARDINAL), 60 (CARDINAL), india (GP...",8
3,GNews,Unknown,cwg 2026 india schedule today july 24 srihari ...,cwg 2026 india schedule today july 24 srihari ...,https://news.google.com/rss/articles/CBMilwFBV...,"india, schedule, today",en,2026-07-24 04:06:45,2026-07-24 04:06:45,2026,7,24,Friday,"india (GPE), today july (DATE), 24 (CARDINAL),...",11
4,GNews,Unknown,in india fear has switched sides al jazeera,in india fear has switched sides al jazeera,https://news.google.com/rss/articles/CBMigwFBV...,"india, fear, switched",en,2026-07-24 04:06:45,2026-07-24 04:06:45,2026,7,24,Friday,"india (GPE), al jazeera (ORG), india (GPE), al...",4


In [13]:
news_df = news.copy()


In [14]:
news_df["title_length"] = news_df["title"].str.split().str.len()


In [15]:
news_df["description_length"] = (
    news_df["description"]
    .str.split()
    .str.len()
)


In [16]:
news_df[
    [
        "title",
        "title_length",
        "description_length"
    ]
].head()


,title,title_length,description_length
0,india dispatch delhis cockroach janta party pr...,10,10
1,returning to play for india kept me motivated ...,11,11
2,us unveils fresh tariffs of up to 125 on 60 ec...,23,23
3,cwg 2026 india schedule today july 24 srihari ...,22,22
4,in india fear has switched sides al jazeera,8,8


In [17]:
news_count = (
    news_df
    .groupby("keyword")
    .size()
    .reset_index(name="news_count")
)


In [18]:
news_count.head()


,keyword,news_count
0,"aamir, khan, showing",1
1,"abhijeet, dipke, indefinite",1
2,"abhijeet, dipke, sonam",2
3,"abhishek, banerjee, quit",1
4,"abhishek, banerjee, voice",1


In [19]:
unique_sources = (
    news_df
    .groupby("keyword")["source_name"]
    .nunique()
    .reset_index(name="unique_sources")
)

unique_sources.head()


,keyword,unique_sources
0,"aamir, khan, showing",1
1,"abhijeet, dipke, indefinite",1
2,"abhijeet, dipke, sonam",1
3,"abhishek, banerjee, quit",1
4,"abhishek, banerjee, voice",1


In [20]:
avg_title_length = (
    news_df
    .groupby("keyword")["title_length"]
    .mean()
    .reset_index(name="avg_title_length")
)

avg_title_length.head()


,keyword,avg_title_length
0,"aamir, khan, showing",15.0
1,"abhijeet, dipke, indefinite",12.0
2,"abhijeet, dipke, sonam",16.5
3,"abhishek, banerjee, quit",12.0
4,"abhishek, banerjee, voice",11.0


In [21]:
news_df["published_date"] = pd.to_datetime(news_df["published_date"])

latest_news = (
    news_df
    .groupby("keyword")["published_date"]
    .max()
    .reset_index(name="latest_news_time")
)

latest_news.head()


,keyword,latest_news_time
0,"aamir, khan, showing",2026-07-17 10:07:45
1,"abhijeet, dipke, indefinite",2026-07-18 04:05:30
2,"abhijeet, dipke, sonam",2026-07-18 08:06:02
3,"abhishek, banerjee, quit",2026-07-19 02:25:22
4,"abhishek, banerjee, voice",2026-07-15 15:04:14


In [22]:
latest_news["news_age_hours"] = (
    pd.Timestamp.now() - latest_news["latest_news_time"]
).dt.total_seconds() / 3600

latest_news.head()


,keyword,latest_news_time,news_age_hours
0,"aamir, khan, showing",2026-07-17 10:07:45,251.788083
1,"abhijeet, dipke, indefinite",2026-07-18 04:05:30,233.825583
2,"abhijeet, dipke, sonam",2026-07-18 08:06:02,229.816694
3,"abhishek, banerjee, quit",2026-07-19 02:25:22,211.494472
4,"abhishek, banerjee, voice",2026-07-15 15:04:14,294.846694


In [23]:
avg_description_length = (
    news_df
    .groupby("keyword")["description_length"]
    .mean()
    .reset_index(name="avg_description_length")
)

avg_description_length.head()


,keyword,avg_description_length
0,"aamir, khan, showing",19.0
1,"abhijeet, dipke, indefinite",24.0
2,"abhijeet, dipke, sonam",24.0
3,"abhishek, banerjee, quit",16.0
4,"abhishek, banerjee, voice",19.0


In [24]:
news_features = news_count.merge(
    unique_sources,
    on="keyword"
)

news_features = news_features.merge(
    avg_title_length,
    on="keyword"
)

news_features = news_features.merge(
    avg_description_length,
    on="keyword"
)

news_features = news_features.merge(
    latest_news,
    on="keyword"
)


In [25]:
news_features.head()


,keyword,news_count,unique_sources,avg_title_length,avg_description_length,latest_news_time,news_age_hours
0,"aamir, khan, showing",1,1,15.0,19.0,2026-07-17 10:07:45,251.788083
1,"abhijeet, dipke, indefinite",1,1,12.0,24.0,2026-07-18 04:05:30,233.825583
2,"abhijeet, dipke, sonam",2,1,16.5,24.0,2026-07-18 08:06:02,229.816694
3,"abhishek, banerjee, quit",1,1,12.0,16.0,2026-07-19 02:25:22,211.494472
4,"abhishek, banerjee, voice",1,1,11.0,19.0,2026-07-15 15:04:14,294.846694


In [27]:
google.head()


,collection_date,collection_time,keyword,latest_interest,rising_queries,country,time_window,source,datetime
0,2026-07-02,15:55:07,second board exam 2026 class 10 result date,43,No Rising Query,india,Past 30 Days,google trends,2026-07-02 15:55:07
1,2026-07-02,15:57:33,ಮಳೆ,72,"ಬೆಳಗಾವಿ ಮಳೆ, ಭವ್ಯಾ ಗೌಡ, ಭಾರತದಲ್ಲಿ ಅತಿ ಹೆಚ್ಚು ಮ...",india,Past 30 Days,google trends,2026-07-02 15:57:33
2,2026-07-02,15:58:44,ఇషాన్ కిషన్,100,No Rising Query,india,Past 30 Days,google trends,2026-07-02 15:58:44
3,2026-07-02,16:03:06,ಮತದಾರ,100,No Rising Query,india,Past 30 Days,google trends,2026-07-02 16:03:06
4,2026-07-02,16:03:38,ಟ್ವೆಂಟಿ೨೦,10,No Rising Query,india,Past 30 Days,google trends,2026-07-02 16:03:38


In [28]:
google.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   collection_date  216 non-null    object
 1   collection_time  216 non-null    object
 2   keyword          216 non-null    object
 3   latest_interest  216 non-null    int64 
 4   rising_queries   216 non-null    object
 5   country          216 non-null    object
 6   time_window      216 non-null    object
 7   source           216 non-null    object
 8   datetime         216 non-null    object
dtypes: int64(1), object(8)
memory usage: 15.3+ KB


In [29]:
google.columns


Index(['collection_date', 'collection_time', 'keyword', 'latest_interest',
       'rising_queries', 'country', 'time_window', 'source', 'datetime'],
      dtype='object')

In [30]:
google["keyword_length"] = google["keyword"].str.split().str.len()

google.head()


,collection_date,collection_time,keyword,latest_interest,rising_queries,country,time_window,source,datetime,keyword_length
0,2026-07-02,15:55:07,second board exam 2026 class 10 result date,43,No Rising Query,india,Past 30 Days,google trends,2026-07-02 15:55:07,8
1,2026-07-02,15:57:33,ಮಳೆ,72,"ಬೆಳಗಾವಿ ಮಳೆ, ಭವ್ಯಾ ಗೌಡ, ಭಾರತದಲ್ಲಿ ಅತಿ ಹೆಚ್ಚು ಮ...",india,Past 30 Days,google trends,2026-07-02 15:57:33,1
2,2026-07-02,15:58:44,ఇషాన్ కిషన్,100,No Rising Query,india,Past 30 Days,google trends,2026-07-02 15:58:44,2
3,2026-07-02,16:03:06,ಮತದಾರ,100,No Rising Query,india,Past 30 Days,google trends,2026-07-02 16:03:06,1
4,2026-07-02,16:03:38,ಟ್ವೆಂಟಿ೨೦,10,No Rising Query,india,Past 30 Days,google trends,2026-07-02 16:03:38,1


In [31]:
google["rising_queries"].iloc[0]


'No Rising Query'

In [32]:
google["rising_queries"].value_counts().head(20)


rising_queries
No Rising Query                                                                                                                                       92
nifty50 hdfc bank market performance, july 7 nifty50 top gainers                                                                                       1
mumbai lake levels news, mumbai lake levels today live, mumbai lake levels latest, mumbai lake levels now, mumbai lake levels twitter                  1
સોનું ભાવ                                                                                                                                              1
nsc india.com, nsc com, nsc interest is taxable or not, nsc portal, 5 year nsc interest rate chart                                                     1
కెనరా బ్యాంకు, బ్యాంకు ఖాతా                                                                                                                            1
येरे येरे पावसा तुला देतो पैसा, तुला पाहते रे, कृष्णा तुला मी ताकीद

In [33]:
def count_queries(text):
    if pd.isna(text) or text == "No Rising Query":
        return 0
    return len(text.split(","))

google["num_rising_queries"] = google["rising_queries"].apply(count_queries)


In [37]:
print(google.columns.tolist())


['collection_date', 'collection_time', 'keyword', 'latest_interest', 'rising_queries', 'country', 'time_window', 'source', 'datetime', 'keyword_length', 'num_rising_queries']


In [38]:
def count_queries(text):
    if pd.isna(text):
        return 0

    text = str(text).strip()

    if text.lower() == "no rising query":
        return 0

    return len([q.strip() for q in text.split(",") if q.strip()])


In [39]:
google.columns


Index(['collection_date', 'collection_time', 'keyword', 'latest_interest',
       'rising_queries', 'country', 'time_window', 'source', 'datetime',
       'keyword_length', 'num_rising_queries'],
      dtype='object')

In [42]:
google.columns


Index(['collection_date', 'collection_time', 'keyword', 'latest_interest',
       'rising_queries', 'country', 'time_window', 'source', 'datetime',
       'keyword_length', 'num_rising_queries'],
      dtype='object')

In [43]:
def count_queries(text):
    if pd.isna(text):
        return 0

    text = str(text).strip()

    if text.lower() == "no rising query":
        return 0

    return len([q.strip() for q in text.split(",") if q.strip()])

google["num_rising_queries"] = google["rising_queries"].apply(count_queries)


In [46]:
youtube = pd.read_csv("data/cleaned/youtube_clean.csv")


In [45]:
youtube.head()


,collection_date,collection_time,keyword,video_id,title,channel_name,published_at,views,likes,comments,tags,video_url
0,2026-07-27,05:15:45,gta| live| gta5| gtav| gaming,X9kX-tMFM74,GTA 5 LIVE #gta5 #gtav #gaming,Hear Is Live,2026-07-27,2098299,4879,1,gta 5 live| gta v live stream| indian train gt...,https://www.youtube.com/watch?v=X9kX-tMFM74
1,2026-07-27,05:15:45,gta| live| don| blink| secret| mod| activated|...,51ZVs8vNsTI,🔴 GTA 5 LIVE – DON’T BLINK! SECRET MOD ACTIVAT...,Sk MAFIA,2026-07-27,694328,5628,0,gia 5 live| gia v live stream| gia 5 gameplay|...,https://www.youtube.com/watch?v=51ZVs8vNsTI
2,2026-07-27,05:15:45,gta| live| gta5| gtav| gaming,alSyObE2IXU,GTA 5 LIVE #gta5 #gtav #gaming,PAJI IS LIVE,2026-07-27,558169,1865,0,No Tags,https://www.youtube.com/watch?v=alSyObE2IXU
3,2026-07-27,05:15:45,gta| live| train| 100| luxury| cars,w3ks5KUhcJ0,GTA 5 LIVE | Train vs 100 Luxury Cars 😱🔥 कौन ब...,Sunil Gaming,2026-07-27,513514,1622,2,gta 5 live| gta v train vs cars| gta 5 experim...,https://www.youtube.com/watch?v=w3ks5KUhcJ0
4,2026-07-27,05:15:45,live| gta| gameplay| raajoo| gaming,x8ngeyuXYX8,🔴LIVE GTA 5 GAMEPLAY | RAAJOO GAMING,Raajoo Gaming,2026-07-27,202035,561,0,GTA 5| gta 5 live| gta v gameplay| gta 5 mod| ...,https://www.youtube.com/watch?v=x8ngeyuXYX8


In [47]:
youtube.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2572 entries, 0 to 2571
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   collection_date  2572 non-null   object
 1   collection_time  1554 non-null   object
 2   keyword          1533 non-null   object
 3   video_id         2572 non-null   object
 4   title            2572 non-null   object
 5   channel_name     2572 non-null   object
 6   published_at     2572 non-null   object
 7   views            2572 non-null   int64 
 8   likes            2572 non-null   int64 
 9   comments         2572 non-null   int64 
 10  tags             2572 non-null   object
 11  video_url        2572 non-null   object
dtypes: int64(3), object(9)
memory usage: 241.2+ KB


In [48]:
youtube_df = youtube.copy()


In [49]:
youtube_df["published_at"] = pd.to_datetime(youtube_df["published_at"])
youtube_df["collection_date"] = pd.to_datetime(youtube_df["collection_date"])


In [50]:
youtube_df.isnull().sum()


collection_date       0
collection_time    1018
keyword            1039
video_id              0
title                 0
channel_name          0
published_at          0
views                 0
likes                 0
comments              0
tags                  0
video_url             0
dtype: int64

In [51]:
youtube[youtube["keyword"].isna()].head(10)


,collection_date,collection_time,keyword,video_id,title,channel_name,published_at,views,likes,comments,tags,video_url
98,2026-07-27,05:15:45,NaN,TU2aTWGCmkM,सोमवार सावन स्पेशल - गौरा संग आओ भोलेनाथ सावन ...,Ganga Devotional,2026-07-26,9479,116,4,गौरा संग आओ भोलेनाथ सावन में| महीना आया सावन क...,https://www.youtube.com/watch?v=TU2aTWGCmkM
150,2026-07-27,05:15:45,NaN,uCGRvzdJir8,शनिवार के दिन जरूर अपने घर में ये सुंदरकांड चल...,Sampurn Sundarkand Path,2026-07-24,158935,1660,147,sunderkand path in 38 minutes| sunderkand path...,https://www.youtube.com/watch?v=uCGRvzdJir8
468,2026-07-24,04:07:04,NaN,NM-GgSKfGT8,शुक्रवार के दिन जरूर अपने घर में ये सुंदरकांड...,Kesari Nandan,2026-07-23,13959,238,22,No Tags,https://www.youtube.com/watch?v=NM-GgSKfGT8
603,2026-07-23,03:59:14,NaN,DcbqJPhxGZ0,गुप्त नवरात्रि अष्टमी के दिन घर में दुर्गा कवच...,Ishwar Katha - ईश्वर कथा,2026-07-21,29999,529,42,durga raksha kavach| maa durga raksha kavach b...,https://www.youtube.com/watch?v=DcbqJPhxGZ0
643,2026-07-23,03:59:14,NaN,4tN6ilxlMVo,हम तुमसे जुदा होकर मर जाएंगे रो रो कर,Kunal Kumar - Topic,2026-07-16,284513,755,0,Kunal Kumar| हम तुमसे जुदा होकर मर जाएंगे रो र...,https://www.youtube.com/watch?v=4tN6ilxlMVo
727,2026-07-22,04:03:58,NaN,2nhWcud8aIE,गुप्त नवरात्रि सप्तमी के दिन घर में दुर्गा कवच...,Ishwar Katha - ईश्वर कथा,2026-07-20,47554,721,48,durga raksha kavach| maa durga raksha kavach b...,https://www.youtube.com/watch?v=2nhWcud8aIE
728,2026-07-22,04:03:58,NaN,h7Dtbo1iUok,मगंलवार के दिन जरूर अपने घर में ये सुंदरकांड च...,Sunder Path,2026-07-20,40688,475,29,No Tags,https://www.youtube.com/watch?v=h7Dtbo1iUok
813,2026-07-21,04:02:55,NaN,jcacLLZGe9c,मगंलवार के दिन जरूर अपने घर में ये सुंदरकांड च...,Divine Sunderkand Path,2026-07-20,17914,218,16,sundarkand path| sundarkand full| hanuman bhaj...,https://www.youtube.com/watch?v=jcacLLZGe9c
915,2026-07-20,05:09:17,NaN,RrZGrUiDiE0,गुप्त नवरात्रि के छठवें दिन घर में दुर्गा कवच ...,Ishwar Katha - ईश्वर कथा,2026-07-19,13478,343,20,durga raksha kavach| maa durga raksha kavach b...,https://www.youtube.com/watch?v=RrZGrUiDiE0
917,2026-07-20,05:09:17,NaN,6l65C3lbO-8,सोमवार के दिन जरूर अपने घर में ये सुंदरकांड चल...,Kesari Nandan,2026-07-19,9772,151,15,No Tags,https://www.youtube.com/watch?v=6l65C3lbO-8


In [52]:
youtube_df = youtube.dropna(subset=["keyword"]).copy()


In [53]:
youtube_df.shape


(1533, 12)

In [54]:
youtube[youtube["keyword"].isna()][
    ["title", "channel_name", "tags"]
].head(10)


,title,channel_name,tags
98,सोमवार सावन स्पेशल - गौरा संग आओ भोलेनाथ सावन ...,Ganga Devotional,गौरा संग आओ भोलेनाथ सावन में| महीना आया सावन क...
150,शनिवार के दिन जरूर अपने घर में ये सुंदरकांड चल...,Sampurn Sundarkand Path,sunderkand path in 38 minutes| sunderkand path...
468,शुक्रवार के दिन जरूर अपने घर में ये सुंदरकांड...,Kesari Nandan,No Tags
603,गुप्त नवरात्रि अष्टमी के दिन घर में दुर्गा कवच...,Ishwar Katha - ईश्वर कथा,durga raksha kavach| maa durga raksha kavach b...
643,हम तुमसे जुदा होकर मर जाएंगे रो रो कर,Kunal Kumar - Topic,Kunal Kumar| हम तुमसे जुदा होकर मर जाएंगे रो र...
727,गुप्त नवरात्रि सप्तमी के दिन घर में दुर्गा कवच...,Ishwar Katha - ईश्वर कथा,durga raksha kavach| maa durga raksha kavach b...
728,मगंलवार के दिन जरूर अपने घर में ये सुंदरकांड च...,Sunder Path,No Tags
813,मगंलवार के दिन जरूर अपने घर में ये सुंदरकांड च...,Divine Sunderkand Path,sundarkand path| sundarkand full| hanuman bhaj...
915,गुप्त नवरात्रि के छठवें दिन घर में दुर्गा कवच ...,Ishwar Katha - ईश्वर कथा,durga raksha kavach| maa durga raksha kavach b...
917,सोमवार के दिन जरूर अपने घर में ये सुंदरकांड चल...,Kesari Nandan,No Tags


In [55]:
youtube_df = youtube.dropna(subset=["keyword"]).copy()


In [56]:
# Total Engagement
youtube_df["engagement"] = youtube_df["likes"] + youtube_df["comments"]

# Engagement Rate
youtube_df["engagement_rate"] = (
    youtube_df["engagement"] / youtube_df["views"]
)

youtube_df["engagement_rate"] = youtube_df["engagement_rate"].fillna(0)

# Video Age
youtube_df["published_at"] = pd.to_datetime(youtube_df["published_at"])
youtube_df["collection_date"] = pd.to_datetime(youtube_df["collection_date"])

youtube_df["video_age_days"] = (
    youtube_df["collection_date"] -
    youtube_df["published_at"]
).dt.days

# Title Length
youtube_df["title_length"] = (
    youtube_df["title"]
    .str.split()
    .str.len()
)


In [57]:
youtube_features = (
    youtube_df
    .groupby("keyword")
    .agg(
        video_count=("video_id", "count"),
        avg_views=("views", "mean"),
        total_views=("views", "sum"),
        avg_likes=("likes", "mean"),
        avg_comments=("comments", "mean"),
        avg_engagement_rate=("engagement_rate", "mean"),
        avg_title_length=("title_length", "mean"),
        latest_video_date=("published_at", "max")
    )
    .reset_index()
)

youtube_features.head()


,keyword,video_count,avg_views,total_views,avg_likes,avg_comments,avg_engagement_rate,avg_title_length,latest_video_date
0,"000, police, bike, gta",1,151792.0,151792,1259.0,22.0,0.008439,8.0,2026-07-14
1,"000, sawblade, car, gta",1,65439.0,65439,891.0,12.0,0.013799,8.0,2026-07-15
2,000| ferrari| opening| jonathan| back| bgmi,1,2995508.0,2995508,239129.0,1002.0,0.080164,11.0,2026-07-16
3,000| handheld| gaming| console,1,69500.0,69500,3253.0,148.0,0.048935,8.0,2026-07-18
4,000| naruto| set| opening| jonathan| back| bgmi,1,3032063.0,3032063,201245.0,739.0,0.066616,11.0,2026-07-20


In [58]:
news_features.head()


,keyword,news_count,unique_sources,avg_title_length,avg_description_length,latest_news_time,news_age_hours
0,"aamir, khan, showing",1,1,15.0,19.0,2026-07-17 10:07:45,251.788083
1,"abhijeet, dipke, indefinite",1,1,12.0,24.0,2026-07-18 04:05:30,233.825583
2,"abhijeet, dipke, sonam",2,1,16.5,24.0,2026-07-18 08:06:02,229.816694
3,"abhishek, banerjee, quit",1,1,12.0,16.0,2026-07-19 02:25:22,211.494472
4,"abhishek, banerjee, voice",1,1,11.0,19.0,2026-07-15 15:04:14,294.846694


In [59]:
google.head()


,collection_date,collection_time,keyword,latest_interest,rising_queries,country,time_window,source,datetime,keyword_length,num_rising_queries
0,2026-07-02,15:55:07,second board exam 2026 class 10 result date,43,No Rising Query,india,Past 30 Days,google trends,2026-07-02 15:55:07,8,0
1,2026-07-02,15:57:33,ಮಳೆ,72,"ಬೆಳಗಾವಿ ಮಳೆ, ಭವ್ಯಾ ಗೌಡ, ಭಾರತದಲ್ಲಿ ಅತಿ ಹೆಚ್ಚು ಮ...",india,Past 30 Days,google trends,2026-07-02 15:57:33,1,5
2,2026-07-02,15:58:44,ఇషాన్ కిషన్,100,No Rising Query,india,Past 30 Days,google trends,2026-07-02 15:58:44,2,0
3,2026-07-02,16:03:06,ಮತದಾರ,100,No Rising Query,india,Past 30 Days,google trends,2026-07-02 16:03:06,1,0
4,2026-07-02,16:03:38,ಟ್ವೆಂಟಿ೨೦,10,No Rising Query,india,Past 30 Days,google trends,2026-07-02 16:03:38,1,0


In [60]:
youtube_features.head()


,keyword,video_count,avg_views,total_views,avg_likes,avg_comments,avg_engagement_rate,avg_title_length,latest_video_date
0,"000, police, bike, gta",1,151792.0,151792,1259.0,22.0,0.008439,8.0,2026-07-14
1,"000, sawblade, car, gta",1,65439.0,65439,891.0,12.0,0.013799,8.0,2026-07-15
2,000| ferrari| opening| jonathan| back| bgmi,1,2995508.0,2995508,239129.0,1002.0,0.080164,11.0,2026-07-16
3,000| handheld| gaming| console,1,69500.0,69500,3253.0,148.0,0.048935,8.0,2026-07-18
4,000| naruto| set| opening| jonathan| back| bgmi,1,3032063.0,3032063,201245.0,739.0,0.066616,11.0,2026-07-20


In [61]:
google["interest_normalized"] = google["latest_interest"] / 100


In [62]:
google["keyword_length"] = google["keyword"].str.split().str.len()


In [63]:
google["datetime"] = pd.to_datetime(google["datetime"])


In [64]:
google["collection_hour"] = google["datetime"].dt.hour


In [65]:
print(google.dtypes)


collection_date                object
collection_time                object
keyword                        object
latest_interest                 int64
rising_queries                 object
country                        object
time_window                    object
source                         object
datetime               datetime64[ns]
keyword_length                  int64
num_rising_queries              int64
interest_normalized           float64
collection_hour                 int32
dtype: object


In [66]:
google["collection_weekday"] = google["datetime"].dt.day_name()


In [67]:
google["collection_weekday"] = google["datetime"].dt.day_name()


In [68]:
def popularity(score):
    if score >= 80:
        return "High"
    elif score >= 50:
        return "Medium"
    else:
        return "Low"

google["popularity"] = google["latest_interest"].apply(popularity)


In [69]:
def count_queries(text):
    if pd.isna(text) or text == "No Rising Query":
        return 0
    return len(text.split(","))

google["num_rising_queries"] = google["rising_queries"].apply(count_queries)


In [70]:
print(google.columns)


Index(['collection_date', 'collection_time', 'keyword', 'latest_interest',
       'rising_queries', 'country', 'time_window', 'source', 'datetime',
       'keyword_length', 'num_rising_queries', 'interest_normalized',
       'collection_hour', 'collection_weekday', 'popularity'],
      dtype='object')


In [71]:
google_features = google[
    [
        "keyword",
        "latest_interest",
        "interest_normalized",
        "keyword_length",
        "collection_hour",
        "collection_weekday",
        "popularity"
    ]
].copy()


In [72]:
google["datetime"] = pd.to_datetime(google["datetime"])


In [73]:
google["keyword_length"] = google["keyword"].str.split().str.len()

google[["keyword", "keyword_length"]].head()


,keyword,keyword_length
0,second board exam 2026 class 10 result date,8
1,ಮಳೆ,1
2,ఇషాన్ కిషన్,2
3,ಮತದಾರ,1
4,ಟ್ವೆಂಟಿ೨೦,1


In [74]:
google["collection_hour"] = google["datetime"].dt.hour

google[["datetime", "collection_hour"]].head()


,datetime,collection_hour
0,2026-07-02 15:55:07,15
1,2026-07-02 15:57:33,15
2,2026-07-02 15:58:44,15
3,2026-07-02 16:03:06,16
4,2026-07-02 16:03:38,16


In [75]:
google["collection_weekday"] = google["datetime"].dt.day_name()

google[["datetime", "collection_weekday"]].head()


,datetime,collection_weekday
0,2026-07-02 15:55:07,Thursday
1,2026-07-02 15:57:33,Thursday
2,2026-07-02 15:58:44,Thursday
3,2026-07-02 16:03:06,Thursday
4,2026-07-02 16:03:38,Thursday


In [76]:
google["interest_normalized"] = google["latest_interest"] / 100

google[
    ["latest_interest", "interest_normalized"]
].head()


,latest_interest,interest_normalized
0,43,0.43
1,72,0.72
2,100,1.00
3,100,1.00
4,10,0.10


In [77]:
def popularity(score):

    if score >= 80:
        return "High"

    elif score >= 40:
        return "Medium"

    else:
        return "Low"

google["popularity"] = google["latest_interest"].apply(popularity)

google[
    ["latest_interest", "popularity"]
].head()


,latest_interest,popularity
0,43,Medium
1,72,Medium
2,100,High
3,100,High
4,10,Low


In [78]:
google_features = google[
    [
        "keyword",
        "latest_interest",
        "interest_normalized",
        "keyword_length",
        "collection_hour",
        "collection_weekday",
        "num_rising_queries",
        "popularity"
    ]
].copy()

google_features.head()


,keyword,latest_interest,interest_normalized,keyword_length,collection_hour,collection_weekday,num_rising_queries,popularity
0,second board exam 2026 class 10 result date,43,0.43,8,15,Thursday,0,Medium
1,ಮಳೆ,72,0.72,1,15,Thursday,5,Medium
2,ఇషాన్ కిషన్,100,1.00,2,15,Thursday,0,High
3,ಮತದಾರ,100,1.00,1,16,Thursday,0,High
4,ಟ್ವೆಂಟಿ೨೦,10,0.10,1,16,Thursday,0,Low


In [79]:
print(news_features.shape)
print(google_features.shape)
print(youtube_features.shape)


(1756, 7)
(216, 8)
(1329, 9)


In [80]:
news_features.columns
google_features.columns
youtube_features.columns


Index(['keyword', 'video_count', 'avg_views', 'total_views', 'avg_likes',
       'avg_comments', 'avg_engagement_rate', 'avg_title_length',
       'latest_video_date'],
      dtype='object')

In [81]:
news_features["keyword"] = (
    news_features["keyword"]
    .astype(str)
    .str.lower()
    .str.strip()
)

google_features["keyword"] = (
    google_features["keyword"]
    .astype(str)
    .str.lower()
    .str.strip()
)

youtube_features["keyword"] = (
    youtube_features["keyword"]
    .astype(str)
    .str.lower()
    .str.strip()
)


In [82]:
news_features["keyword"].head()
google_features["keyword"].head()
youtube_features["keyword"].head()


0                             000, police, bike, gta
1                            000, sawblade, car, gta
2        000| ferrari| opening| jonathan| back| bgmi
3                     000| handheld| gaming| console
4    000| naruto| set| opening| jonathan| back| bgmi
Name: keyword, dtype: object

In [83]:
news_features = news_features.drop_duplicates("keyword")

google_features = google_features.drop_duplicates("keyword")

youtube_features = youtube_features.drop_duplicates("keyword")


In [84]:
master = pd.merge(
    news_features,
    google_features,
    on="keyword",
    how="outer"
)


In [85]:
master.head()

master.shape


(1967, 14)

In [86]:
master = pd.merge(
    master,
    youtube_features,
    on="keyword",
    how="outer"
)


In [87]:
master.head()

master.shape

master.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3296 entries, 0 to 3295
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   keyword                 3296 non-null   object        
 1   news_count              1756 non-null   float64       
 2   unique_sources          1756 non-null   float64       
 3   avg_title_length_x      1756 non-null   float64       
 4   avg_description_length  1756 non-null   float64       
 5   latest_news_time        1756 non-null   datetime64[ns]
 6   news_age_hours          1756 non-null   float64       
 7   latest_interest         211 non-null    float64       
 8   interest_normalized     211 non-null    float64       
 9   keyword_length          211 non-null    float64       
 10  collection_hour         211 non-null    float64       
 11  collection_weekday      211 non-null    object        
 12  num_rising_queries      211 non-null    float64 

In [88]:
master.isnull().sum()


keyword                      0
news_count                1540
unique_sources            1540
avg_title_length_x        1540
avg_description_length    1540
latest_news_time          1540
news_age_hours            1540
latest_interest           3085
interest_normalized       3085
keyword_length            3085
collection_hour           3085
collection_weekday        3085
num_rising_queries        3085
popularity                3085
video_count               1967
avg_views                 1967
total_views               1967
avg_likes                 1967
avg_comments              1967
avg_engagement_rate       1967
avg_title_length_y        1967
latest_video_date         1967
dtype: int64

In [89]:
master.head(10)


,keyword,news_count,unique_sources,avg_title_length_x,avg_description_length,latest_news_time,news_age_hours,latest_interest,interest_normalized,keyword_length,...,num_rising_queries,popularity,video_count,avg_views,total_views,avg_likes,avg_comments,avg_engagement_rate,avg_title_length_y,latest_video_date
0,"000, police, bike, gta",NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,151792.0,151792.0,1259.0,22.0,0.008439,8.0,2026-07-14
1,"000, sawblade, car, gta",NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,65439.0,65439.0,891.0,12.0,0.013799,8.0,2026-07-15
2,000| ferrari| opening| jonathan| back| bgmi,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,2995508.0,2995508.0,239129.0,1002.0,0.080164,11.0,2026-07-16
3,000| handheld| gaming| console,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,69500.0,69500.0,3253.0,148.0,0.048935,8.0,2026-07-18
4,000| naruto| set| opening| jonathan| back| bgmi,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,3032063.0,3032063.0,201245.0,739.0,0.066616,11.0,2026-07-20
5,000| rocket| car| gta,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,35345.0,35345.0,284.0,0.0,0.008035,8.0,2026-07-22
6,1,NaN,NaN,NaN,NaN,NaT,NaN,64.0,0.64,1.0,...,5.0,Medium,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
7,"1000, camouflage, hide, seek, mecha, chameleon",NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,72224.0,72224.0,3701.0,258.0,0.054816,9.0,2026-07-14
8,100000| ferrari| crate| opening| jokerkihaveli,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,288079.0,288079.0,19521.0,15.0,0.067815,6.0,2026-07-17
9,100| 000| spider| man| gta,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,506984.0,506984.0,4408.0,80.0,0.008852,8.0,2026-07-19


In [90]:
master.sample(10)


,keyword,news_count,unique_sources,avg_title_length_x,avg_description_length,latest_news_time,news_age_hours,latest_interest,interest_normalized,keyword_length,...,num_rising_queries,popularity,video_count,avg_views,total_views,avg_likes,avg_comments,avg_engagement_rate,avg_title_length_y,latest_video_date
2064,nas| kyu| dukhe| anjali| raghav| karan| chaudh...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,3.0,686653.0,2059959.0,1806.0,27.333333,0.002721,19.666667,2026-07-19
1044,free| fire| malayalam| rank| season| push| live,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,111109.0,111109.0,3602.0,2.000000,0.032437,10.000000,2026-07-18
673,"delhi, rain, gets",1.0,1.0,7.0,7.0,2026-07-09 14:20:17,439.579194,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2534,"ronaldo, messi, kaif",1.0,1.0,10.0,59.0,2026-07-15 12:45:28,297.159472,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2117,"nurse, proposes, lover",1.0,1.0,11.0,18.0,2026-07-20 02:30:24,187.410583,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2908,"ties, continuity, andy",1.0,1.0,13.0,13.0,2026-07-19 03:42:38,210.206694,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2760,"spokesperson, dahiya, hungry",1.0,1.0,17.0,20.0,2026-07-21 10:59:26,154.926694,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
3105,"wangchuk, hunger, strike",1.0,1.0,14.0,26.0,2026-07-17 02:11:08,259.731694,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
876,fifa world cup final,NaN,NaN,NaN,NaN,NaT,NaN,100.0,1.0,4.0,...,1.0,High,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2695,sojugada| sooju| mallige| shiva| kannada| bhak...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,22711.0,22711.0,122.0,1.000000,0.005416,14.000000,2026-07-19


In [91]:
master.rename(
    columns={
        "avg_title_length_x": "avg_news_title_length",
        "avg_title_length_y": "avg_video_title_length"
    },
    inplace=True
)


In [92]:
master["platform_count"] = (
    master["news_count"].notna().astype(int)
    + master["latest_interest"].notna().astype(int)
    + master["video_count"].notna().astype(int)
)


In [93]:
master["has_news"] = master["news_count"].notna().astype(int)


In [94]:
master["has_google"] = master["latest_interest"].notna().astype(int)


In [95]:
master["has_youtube"] = master["video_count"].notna().astype(int)


In [96]:
master["attention_score"] = (
    master["news_count"].fillna(0)
    + master["latest_interest"].fillna(0)
    + master["video_count"].fillna(0)
)


In [97]:
master["fresh_news"] = (
    master["news_age_hours"] < 24
).astype(int)


In [98]:
master["high_google_interest"] = (
    master["latest_interest"] >= 80
).astype(int)


In [99]:
master["viral_video"] = (
    master["avg_engagement_rate"] >= 0.05
).astype(int)


In [100]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

master[
    [
        "news_count",
        "latest_interest",
        "video_count",
        "avg_views"
    ]
] = scaler.fit_transform(
    master[
        [
            "news_count",
            "latest_interest",
            "video_count",
            "avg_views"
        ]
    ].fillna(0)
)


In [101]:
master["news_score"] = (
    0.6 * master["news_count"] +
    0.4 * master["unique_sources"].fillna(0)
)


In [192]:
master["google_score"] = (
    0.7 * master["latest_interest"] +
    0.3 * master["num_rising_queries"].fillna(0)
)


In [193]:
master["youtube_score"] = (
    0.5 * master["video_count"] +
    0.5 * master["avg_engagement_rate"].fillna(0)
)


In [194]:
master["trend_score"] = (
    0.30 * master["news_score"] +
    0.50 * master["google_score"] +
    0.20 * master["youtube_score"]
)


In [195]:
master = master.sort_values(
    "trend_score",
    ascending=False
)

master.head(20)


,keyword,news_count,unique_sources,avg_news_title_length,avg_description_length,latest_news_time,news_age_hours,latest_interest,interest_normalized,keyword_length,...,has_google,has_youtube,attention_score,fresh_news,high_google_interest,viral_video,news_score,google_score,youtube_score,trend_score
3257,ਮੌਸਮ,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,1.0,...,1,0,100.0,0,1,0,0.0,2.8,0.0,1.4
3269,பணம்,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,1.0,...,1,0,100.0,0,1,0,0.0,2.2,0.0,1.1
1763,lautaro martínez,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,2.0,...,1,0,100.0,0,1,0,0.0,2.2,0.0,1.1
2583,sarnath,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,1.0,...,1,0,100.0,0,1,0,0.0,2.2,0.0,1.1
2036,mumbai lake levels,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,3.0,...,1,0,100.0,0,1,0,0.0,2.2,0.0,1.1
692,devendra fadnavis,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,2.0,...,1,0,100.0,0,1,0,0.0,2.2,0.0,1.1
3240,राजनीति,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,1.0,...,1,0,100.0,0,1,0,0.0,2.2,0.0,1.1
1739,la rosa de guadalupe,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,4.0,...,1,0,100.0,0,1,0,0.0,2.2,0.0,1.1
288,bahuda yatra,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,2.0,...,1,0,100.0,0,1,0,0.0,2.2,0.0,1.1
2067,nations league,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,2.0,...,1,0,100.0,0,1,0,0.0,2.2,0.0,1.1


In [196]:
master["trend_rank"] = range(1, len(master) + 1)


In [197]:
master.to_csv(
    "../data/features/master_trend_features.csv",
    index=False
)


In [102]:
import os

os.makedirs("../data/features", exist_ok=True)

master.to_csv(
    "../data/features/master_trend_features.csv",
    index=False
)


In [199]:
master.shape


(3296, 35)

In [200]:
master.columns


Index(['keyword', 'news_count', 'unique_sources', 'avg_news_title_length',
       'avg_description_length', 'latest_news_time', 'news_age_hours',
       'latest_interest', 'interest_normalized', 'keyword_length',
       'collection_hour', 'collection_weekday', 'num_rising_queries',
       'popularity', 'video_count', 'avg_views', 'total_views', 'avg_likes',
       'avg_comments', 'avg_engagement_rate', 'avg_video_title_length',
       'latest_video_date', 'platform_count', 'has_news', 'has_google',
       'has_youtube', 'attention_score', 'fresh_news', 'high_google_interest',
       'viral_video', 'news_score', 'google_score', 'youtube_score',
       'trend_score', 'trend_rank'],
      dtype='object')

In [201]:
master.columns


Index(['keyword', 'news_count', 'unique_sources', 'avg_news_title_length',
       'avg_description_length', 'latest_news_time', 'news_age_hours',
       'latest_interest', 'interest_normalized', 'keyword_length',
       'collection_hour', 'collection_weekday', 'num_rising_queries',
       'popularity', 'video_count', 'avg_views', 'total_views', 'avg_likes',
       'avg_comments', 'avg_engagement_rate', 'avg_video_title_length',
       'latest_video_date', 'platform_count', 'has_news', 'has_google',
       'has_youtube', 'attention_score', 'fresh_news', 'high_google_interest',
       'viral_video', 'news_score', 'google_score', 'youtube_score',
       'trend_score', 'trend_rank'],
      dtype='object')

In [202]:
master.head()


,keyword,news_count,unique_sources,avg_news_title_length,avg_description_length,latest_news_time,news_age_hours,latest_interest,interest_normalized,keyword_length,...,has_youtube,attention_score,fresh_news,high_google_interest,viral_video,news_score,google_score,youtube_score,trend_score,trend_rank
3257,ਮੌਸਮ,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,1.0,...,0,100.0,0,1,0,0.0,2.8,0.0,1.4,1
3269,பணம்,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,1.0,...,0,100.0,0,1,0,0.0,2.2,0.0,1.1,2
1763,lautaro martínez,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,2.0,...,0,100.0,0,1,0,0.0,2.2,0.0,1.1,3
2583,sarnath,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,1.0,...,0,100.0,0,1,0,0.0,2.2,0.0,1.1,4
2036,mumbai lake levels,0.0,NaN,NaN,NaN,NaT,NaN,1.0,1.0,3.0,...,0,100.0,0,1,0,0.0,2.2,0.0,1.1,5
